In [1]:
import os
os.environ.setdefault("USER_AGENT", "AI Agents and Agentic Workflows educational RAG notebook")

from langchain_community.document_loaders import WikipediaLoader, WebBaseLoader, Docx2txtLoader, PyPDFLoader, TextLoader, DirectoryLoader

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama

## Setting up vector database and embeddings

In [2]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
embeddings_model = None  # Use Chroma's local all-MiniLM-L6-v2 embeddings
vector_db = Chroma("tourist_info", embeddings_model)

> **Chroma defaults used here (verified with the installed Chroma 1.3.0):** For a newly created local/single-node collection, `embeddings_model = None` lets Chroma attach its built-in `DefaultEmbeddingFunction`, which uses ONNX Runtime with `all-MiniLM-L6-v2`. The main dense-vector index is **HNSW** (approximate nearest-neighbor search), not IVF or PQ. Its default `space` is **`l2`**, which Chroma defines as squared Euclidean distance: $\sum_i (A_i-B_i)^2$. Therefore, smaller returned distances mean closer matches; this is not cosine similarity or dot product. Newly added vectors first enter a small brute-force buffer (default batch size: 100) before being merged into HNSW. These settings are established when the collection is created.

> Sources: [Chroma index configuration](https://docs.trychroma.com/docs/collections/configure) and [Chroma collection defaults](https://cookbook.chromadb.dev/core/collections/).

In [3]:
try:
    wikipedia_loader = WikipediaLoader(query="Paestum")
    wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
    vector_db.add_documents(wikipedia_chunks)
except Exception as error:
    print(f"Wikipedia API failed ({type(error).__name__}: {error}). Loading the Paestum page directly.")
    wikipedia_loader = WebBaseLoader(
        "https://en.wikipedia.org/wiki/Paestum",
        header_template={"User-Agent": "AI Agents and Agentic Workflows educational RAG notebook"}
    )
    wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
    vector_db.add_documents(wikipedia_chunks)

Wikipedia API failed (JSONDecodeError: Expecting value: line 1 column 1 (char 0)). Loading the Paestum page directly.


In [4]:
word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_chunks = text_splitter.split_documents(word_loader.load())
vector_db.add_documents(word_chunks)

['fe00476b-2547-434d-93af-cf672b719bd2',
 'c9932372-49f2-4628-b240-63967dd13069',
 '7a702712-6e47-4045-9ec0-1f9a47228acc',
 '6c7c3fe3-00cc-4e5d-8d1b-7beb05eb51b3',
 '861d0961-0c41-41e0-bdae-51744bd1cb21',
 'bd247acd-90f0-4f4f-a6e4-b327dd828065',
 'aaa1ab1b-c312-4fcd-8a47-f3725245e46f',
 '30230b00-683a-4619-8de4-72468e439bbb']

In [5]:
pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_chunks = text_splitter.split_documents(pdf_loader.load())
vector_db.add_documents(pdf_chunks)

['26f14f77-4ed3-4f83-9e07-a0c01e041bc5',
 '2eaebbd8-9f86-4c8e-9312-7be82f76eb4a',
 'c8bd5cdb-625f-43ad-b43b-0c99a28d961f',
 '5d784581-c8c5-4b5d-ad11-95453aa0d58b',
 'd528c4bc-fc4d-49fa-8510-7144c319d4f0',
 '39c9cc4e-7512-4beb-9505-6af73b6377bb',
 'ac27b329-c18e-4f53-ab61-cfcc3cd1e726',
 '96617261-f4aa-4fec-bcd5-5571ebd40992',
 '4826adb9-9a16-4bee-bc4d-4f0f23965d4e',
 'd7531889-13aa-4c13-9a3b-bd282f2e2b42',
 'b5bcfc45-f5e0-45f5-adb9-5461ac0c029f',
 '309102c4-2a66-4d37-b9da-088b85a164f4',
 'f6921aff-8b1a-4e1b-b305-c2b156f19a5a',
 'adee1ca8-f4d2-4def-99b0-54eb247a28fc',
 '904f552c-54da-45db-a45b-b1af8e93fdcf',
 'd32e62e4-5ba6-430a-bc93-fa5694765c02',
 '67dd1db4-f0b2-40ba-9fac-cea7be765ada',
 '286dbdb1-2cea-47d7-b6b6-3624157b9947',
 '65c78ca5-ddbd-4b0e-883f-39008ee9b5a1',
 '628f502a-a219-4293-9d38-d8ae9d46ad6f',
 'a3ce8698-7a6a-48af-8ccf-a6656f5f27b1',
 '308b2415-7c7d-4ee7-a7ee-411e897f2c2b',
 '2bc776f4-1662-4a5a-b323-a885e37b02b0',
 '7870d384-a10d-4209-bbc5-766f290f2068',
 'de9b252d-3135-

In [6]:
txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_chunks = text_splitter.split_documents(txt_loader.load())
vector_db.add_documents(txt_chunks)

['4e7ee61f-5c8f-4cf9-8946-c8d476c80dd9']

## Removing duplication

In [7]:
def split_and_import(loader):
     chunks = text_splitter.split_documents(loader.load())
     vector_db.add_documents(chunks)
     print(f"Ingested chunks created by {loader}")

In [8]:
try:
    wikipedia_loader = WikipediaLoader(query="Paestum")
    split_and_import(wikipedia_loader)
except Exception as error:
    print(f"Wikipedia API failed ({type(error).__name__}: {error}). Loading the Paestum page directly.")
    wikipedia_loader = WebBaseLoader(
        "https://en.wikipedia.org/wiki/Paestum",
        header_template={"User-Agent": "AI Agents and Agentic Workflows educational RAG notebook"}
    )
    split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

Wikipedia API failed (JSONDecodeError: Expecting value: line 1 column 1 (char 0)). Loading the Paestum page directly.
Ingested chunks created by <langchain_community.document_loaders.web_base.WebBaseLoader object at 0x00000209275A7610>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x000002092546DD10>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x000002092546EE90>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x00000209257E4550>


## Ingesting Multiple Documents from a Folder (two techniques)

### 1) Iterating over all files in a folder

In [9]:
loader_classes = {
    'docx': Docx2txtLoader,
    'pdf': PyPDFLoader,
    'txt': TextLoader
}

In [10]:
import os

def get_loader(filename):
    _, file_extension = os.path.splitext(filename) #A Extract the file extension
    file_extension = file_extension.lstrip('.') #B Remove the leading dot from the extension

    loader_class = loader_classes.get(
        file_extension) #C Get the loader class from the dictionary

    if loader_class:
        return loader_class(filename) #D Instantiate and return the correct loader
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

### Ingesting the files from the folder

In [11]:
folder_path = "CilentoTouristInfo" #A Path to the folder containing the documents

for filename in os.listdir(folder_path): #B iterate over the files in the path
    file_path = os.path.join(folder_path, filename) #C Construct the full path to the file

    if os.path.isfile(file_path): #D Check if it is a file (not a directory)
        try:
            loader = get_loader(file_path) #E Instantiate the correct loader for the file
            print(f"Loader for {filename}: {loader}")
            split_and_import(loader) #F Split and ingest
        except ValueError as e:
            print(e)

Loader for Acciaroli.pdf: <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x00000209257E47D0>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x00000209257E47D0>
Loader for Cape Palinuro.txt: <langchain_community.document_loaders.text.TextLoader object at 0x000002092546FC50>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x000002092546FC50>
Loader for Casalvelino.txt: <langchain_community.document_loaders.text.TextLoader object at 0x000002092549EEA0>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x000002092549EEA0>
Loader for Cilentan coast.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x00000209257E47D0>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x00000209257E47D0>
Loader for Cilento Coast Map and Travel Guide.docx: <langchain_community.docum

### 2) Ingesting all files with DirectoryLoader

In [12]:
# Unstructured's local PDF inference extra does not support Windows Python 3.13.
# Use Unstructured locally for DOCX/TXT and the existing PyPDFLoader for PDF files.
# Requires: unstructured[docx]
# https://docs.langchain.com/oss/python/integrations/providers/unstructured
# https://docs.unstructured.io/open-source/installation/full-installation
folder_path = "CilentoTouristInfo"

unstructured_directory_loader = DirectoryLoader(
    folder_path, ["**/*.docx", "**/*.txt"]
) #A Load DOCX and TXT files with Unstructured
pdf_directory_loader = DirectoryLoader(
    folder_path, "**/*.pdf", loader_cls=PyPDFLoader
) #B Load PDFs locally with PyPDFLoader

split_and_import(unstructured_directory_loader)
split_and_import(pdf_directory_loader)

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


Ingested chunks created by <langchain_community.document_loaders.directory.DirectoryLoader object at 0x0000020925BB3CB0>
Ingested chunks created by <langchain_community.document_loaders.directory.DirectoryLoader object at 0x00000209257E4410>


> **Windows/Python compatibility note:** Unstructured's local PDF inference dependency is not available for Windows Python 3.13. Therefore, this notebook uses Unstructured locally for DOCX/TXT files and the existing `PyPDFLoader` for PDFs. For full Unstructured PDF/OCR processing, use Python 3.12, install the PDF extra, and provide the required system tools such as Poppler and Tesseract. See the [LangChain Unstructured integration](https://docs.langchain.com/oss/python/integrations/providers/unstructured) and [Unstructured full-installation guide](https://docs.unstructured.io/open-source/installation/full-installation).
>
> Repeated `libmagic is unavailable` messages are non-fatal file-type-detection advisories. This notebook supplies explicit `.docx`, `.txt`, and `.pdf` glob patterns, so files with correct extensions can still be loaded without native `libmagic`. The two `Ingested chunks created by ... DirectoryLoader` messages confirm that both loader paths—Unstructured for DOCX/TXT and `PyPDFLoader` for PDF—completed successfully. Native `libmagic` is mainly useful here for files with missing, incorrect, or ambiguous extensions.

## Querying the vector store directly

In [13]:
query = "Where was Poseidonia and who renamed it to Paestum?"
results = vector_db.similarity_search(query, 4) # four clostest results
print(results)

[Document(id='66fee75e-c3e4-4666-a44d-f393f42397ce', metadata={'source': 'https://en.wikipedia.org/wiki/Paestum', 'title': 'Paestum - Wikipedia', 'language': 'en'}, page_content='The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its current name.[5]\nAncient ruins and features[edit]\nAerial view of Paestum, looking north; two Hera Temples in foreground, Athena Temple in background.'), Document(id='40dc8222-310c-4aab-8142-968a06bd4918', metadata={'title': 'Paestum - Wikipedia', 'source': 'https://en.wikipedia.org/wiki/Paestum', 'language': 'en'}, page_content='The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its curr

In [14]:
len(results)

4

## Asking a question through a LangChain's RAG chain

In [15]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """Use the following pieces of context
to answer the question at the end.
If you don't know the answer, just say that you don't know,
don't try to make up an answer.
Use three sentences maximum and keep the
answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

Alternatively, you can pull the prompt instance directly from the **[LangChain Hub](https://smith.langchain.com/hub)**:

```Python
from langchain import hub
rag_prompt = hub.pull("rlm/rag-prompt")
```

In [16]:
retriever = vector_db.as_retriever()

In [17]:
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()

`RunnablePassthrough` is a core component in LangChain's LangChain Reference Expression Language (LCEL) that _takes an input and **returns it completely unchanged**_. It acts like an identity function, making it essential for passing raw data—like a user's original question—alongside processed intermediate data in complex multi-step chains.
- Retrieval-Augmented Generation (RAG) Use Case: Pass the original user query straight through to a prompt template while a separate retriever branch fetches context.

In [18]:
chatbot = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=65536,
    temperature=0,
    reasoning=False
)

In [19]:
# set up RAG chain

rag_chain = {"context": retriever,
             "question": question_feeder} | rag_prompt | chatbot

In [20]:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [21]:
question = """Where was Poseidonia and who renamed
it to Paestum. Also tell me the source."""
answer = execute_chain(rag_chain, question)
print(answer.content)

Poseidonia was the original name given to the city by Greek settlers, which is now known as Paestum. The Romans gave the city its current name after it was conquered by the Lucanians, who had renamed it Paistos. The source for this information is Wikipedia.


In [22]:
print(answer)

content='Poseidonia was the original name given to the city by Greek settlers, which is now known as Paestum. The Romans gave the city its current name after it was conquered by the Lucanians, who had renamed it Paistos. The source for this information is Wikipedia.' additional_kwargs={} response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-08T05:10:10.7523278Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7271619800, 'load_duration': 5577784800, 'prompt_eval_count': 691, 'prompt_eval_duration': 300934000, 'eval_count': 57, 'eval_duration': 1387286000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'} id='lc_run--019fdfc7-39d7-7bc0-ab53-f6388ffbc320-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 691, 'output_tokens': 57, 'total_tokens': 748}


In [23]:
question = """And then, what they do?
Tell me only if you know.
Also tell me the source"""
answer = execute_chain(rag_chain, question)
print(answer.content)

I do not know what they do because the provided documents do not contain information regarding their actions or activities. The sources include "CilentoTouristInfo\Parmenides.docx" and Wikipedia pages for Paestum.


## Chatbot memory of message history

In [24]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible."),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=65536,
    temperature=0,
    reasoning=False
)
chat_history_memory = ChatMessageHistory()

def get_messages(x):
    return chat_history_memory.messages

rag_chain = {
    "retrieved_context": retriever,
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
} | rag_prompt | chatbot

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')
    return answer

In [25]:
question = """Where was Poseidonia and who renamed
it to Paestum? Also tell me the source."""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed\nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was founded by Greek settlers in the location now known as Paestum. The local Lucanians renamed it to Paistos, while the Romans eventually gave the city its current name. This information is sourced from Wikipedia.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-08T05:10:15.172798Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1946992600, 'load_duration': 318792000, 'prompt_eval_count': 748, 'prompt_eval_duration': 287641000, 'eval_count': 47, 'eval_duration': 1100612000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'}, id='lc_run--019fdfc7-5fe8-7240-b046-8277c49466a0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 748, 'output_tokens': 47, 'total_tokens': 795})]


P

In [26]:
question = """And then what did they do?
Also tell me the source"""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed\nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was founded by Greek settlers in the location now known as Paestum. The local Lucanians renamed it to Paistos, while the Romans eventually gave the city its current name. This information is sourced from Wikipedia.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-08T05:10:15.172798Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1946992600, 'load_duration': 318792000, 'prompt_eval_count': 748, 'prompt_eval_duration': 287641000, 'eval_count': 47, 'eval_duration': 1100612000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'}, id='lc_run--019fdfc7-5fe8-7240-b046-8277c49466a0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 748, 'output_tokens': 47, 'total_tokens': 795}), Hum

## Tracing with LangSmith

Stop the notebook and open a new operative system shell (for example Windows command shell).

Configure the relevant environment variables in the OS shell, the rerun the previous Jupyter cells:
```
(env_ch07) C:\...\ch07>set LANGSMITH_TRACING=true
(env_ch07) C:\...\ch07>set LANGSMITH_ENDPOINT=https://api.smith.langchain.com
(env_ch07) C:\...\ch07>set LANGSMITH_PROJECT=Q & A chatbot
(env_ch07) C:\...\ch07>set LANGSMITH_API_KEY=<YOUR_LANGSMITH_API_KEY>
```
Then Restart the Jupyter notebook:

```(env_ch07) C:\...\ch07>jupyter notebook 07-QA_across_documents.ipynb```

Finally re-execute the whole Jupyter notebook cell by cell. All the activity will have not been logged through LangSmith.